In [75]:
import pyspark
import os
import sys
from pyspark.sql import SparkSession

# Notebook'un kullandığı Python yolunu al
python_path = sys.executable
print("Using Python:", python_path)

# Driver ve worker aynı Python'u kullansın
os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

# Eski session varsa kapat
try:
    spark.stop()
except:
    pass

# Yeni session başlat
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .config("spark.pyspark.python", python_path) \
    .config("spark.pyspark.driver.python", python_path) \
    .getOrCreate()

print("Spark started")

Using Python: /Users/selcukkaleli/data-engineering-zoomcamp/.venv/bin/python3
Spark started


In [5]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

--2026-03-07 18:05:30--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/035746e8-4e24-47e8-a3ce-edcf6d1b11c7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-07T15%3A47%3A22Z&rscd=attachment%3B+filename%3Dfhvhv_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-07T14%3A47%3A08Z&ske=2026-03-07T15%3A47%3A22Z&sks=b&skv=2018-11-09&sig=CmzvMynUb67NVJAjVyKgna%2FvCfnkcZ5xg8GCw9t0QO0%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3Mjg5OTUzMCwibmJmIjoxNzcyODk1OTMwLCJwYXRoIj

In [3]:
!gunzip -c fhvhv_tripdata_2021-01.csv.gz > fhvhv_tripdata_2021-01.csv

In [4]:
!wc -l fhvhv_tripdata_2021-01.csv

 11908469 fhvhv_tripdata_2021-01.csv


In [82]:
df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv')

In [83]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [84]:
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [85]:
!head -n 10 head.csv

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,
HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,
HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,
HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,
HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,
HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,
HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,
HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,
HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,


In [86]:
import pandas as pd

In [87]:
df_pandas = pd.read_csv('head.csv')

In [88]:
df_pandas.dtypes

hvfhs_license_num        object
dispatching_base_num     object
pickup_datetime          object
dropoff_datetime         object
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [89]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [90]:
from pyspark.sql import types

In [91]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [92]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

In [93]:
df = df.repartition(24)

In [94]:
df.write.mode("overwrite").parquet('fhvhv/2021/01/')

26/03/07 19:38:09 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/07 19:38:09 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/07 19:38:09 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/07 19:38:09 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/07 19:38:09 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/07 19:38:09 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/07 19:38:09 WARN MemoryManager: Total allocation exceeds 95.

In [97]:
df = spark.read.parquet('fhvhv/2021/01/')

In [98]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [99]:
from pyspark.sql import functions as F

In [100]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [101]:
crazy_stuff('B02884')

's/b44'

In [102]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [103]:
df\
    .withColumn('pickup_date',F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date',F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID','DOLocationID') \
    .show()

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  s/acd| 2021-01-02|  2021-01-02|         237|         234|
|  e/acc| 2021-01-01|  2021-01-01|         231|         151|
|  e/b32| 2021-01-01|  2021-01-01|         247|          31|
|  e/9ce| 2021-01-01|  2021-01-01|          94|         244|
|  e/b32| 2021-01-03|  2021-01-03|          72|          37|
|  e/95b| 2021-01-02|  2021-01-02|         215|          28|
|  e/9ce| 2021-01-01|  2021-01-01|          61|          39|
|  e/9ce| 2021-01-01|  2021-01-01|         223|         179|
|  e/9ce| 2021-01-01|  2021-01-01|         146|         140|
|  s/b44| 2021-01-03|  2021-01-03|          76|          76|
|  e/9ce| 2021-01-01|  2021-01-01|          71|          63|
|  a/b31| 2021-01-01|  2021-01-01|          26|          11|
|  e/9ce| 2021-01-01|  2021-01-01|         171|          16|
|  e/b38| 2021-01-01|  2

In [104]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID','DOLocationID') \
.filter(df.hvfhs_license_num == 'HV0003') \
.show()

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-02 20:40:11|2021-01-02 20:51:33|         237|         234|
|2021-01-01 01:18:19|2021-01-01 01:38:45|         231|         151|
|2021-01-01 01:36:01|2021-01-01 01:50:23|         247|          31|
|2021-01-03 12:46:48|2021-01-03 12:57:42|          72|          37|
|2021-01-02 13:41:12|2021-01-02 13:49:34|         215|          28|
|2021-01-03 00:31:31|2021-01-03 00:38:24|          76|          76|
|2021-01-01 02:01:18|2021-01-01 02:10:51|          26|          11|
|2021-01-01 15:13:03|2021-01-01 15:31:50|          25|          85|
|2021-01-01 06:06:07|2021-01-01 06:20:00|          89|          25|
|2021-01-01 02:31:22|2021-01-01 02:34:50|          69|         169|
|2021-01-01 22:37:48|2021-01-01 22:44:31|         129|         260|
|2021-01-01 12:22:27|2021-01-01 12:37:01|       